# PROPOSED

In [ ]:
import time
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from functools import lru_cache
from scipy.integrate import cumulative_trapezoid
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 0. PENGATURAN REPRODUKSIBILITAS & CPU (PENTING UNTUK KAGGLE)
# ==========================================
torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cpu")
# Mengunci thread CPU ke-1 untuk latensi yang deterministik dan stabil
torch.set_num_threads(1) 

# ==========================================
# 1. PERSIAPAN DATASET REAL (ADULT INCOME)
# ==========================================
print("Mengunduh dataset Adult Income dari OpenML...")
X_raw, y_raw = fetch_openml(name='adult', version=2, return_X_y=True, as_frame=True)
df = pd.concat([X_raw, y_raw], axis=1).dropna()

X_df = df.iloc[:, :-1]
y_df = df.iloc[:, -1]

# Binerisasi target (>50K)
y_binary = (y_df.astype(str).str.contains('>50K')).astype(int).values
X_encoded = pd.get_dummies(X_df, drop_first=True).astype(float).values

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y_binary, test_size=0.2, random_state=42, stratify=y_binary
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

input_dim = X_train_scaled.shape[1]
print(f"Dimensi Input: {input_dim}")
print(f"Jumlah data Train: {len(X_train_tensor):,}")
print(f"Jumlah data Test: {len(X_test_tensor):,}")

# ==========================================
# 2. ARSITEKTUR BASELINE (WIDE MLP)
# ==========================================
class WideMLP(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, 1024)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(1024, 512)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(512, 256)
        self.relu3 = nn.ReLU()
        self.head = nn.Linear(256, 1)

    def forward(self, x):
        x = self.relu1(self.fc1(x))
        x = self.relu2(self.fc2(x))
        x = self.relu3(self.fc3(x))
        return self.head(x)

base_model = WideMLP(input_dim)
optimizer = optim.Adam(base_model.parameters(), lr=0.002)
criterion = nn.BCEWithLogitsLoss()

print("\n[1/3] Melatih Baseline Model (15 Epochs)...")
base_model.train()
batch_size_train = 256
n_samples = len(X_train_tensor)

for epoch in range(15):
    permutation = torch.randperm(n_samples)
    for i in range(0, n_samples, batch_size_train):
        indices = permutation[i:i+batch_size_train]
        batch_x, batch_y = X_train_tensor[indices], y_train_tensor[indices]
        
        optimizer.zero_grad()
        loss = criterion(base_model(batch_x), batch_y)
        loss.backward()
        optimizer.step()

base_model.eval()

# ==========================================
# 3. METODE USULAN: MP-TAIL (q=0.25) + SEQ BIAS
# ==========================================
class FactoredLinear(nn.Module):
    def __init__(self, W_A, W_B, bias_adj):
        super().__init__()
        k, in_f = W_A.shape
        out_f, _ = W_B.shape
        self.fc1 = nn.Linear(in_f, k, bias=False)
        self.fc2 = nn.Linear(k, out_f, bias=True)
        
        with torch.no_grad():
            self.fc1.weight.copy_(W_A)
            self.fc2.weight.copy_(W_B)
            self.fc2.bias.copy_(bias_adj)

    def forward(self, x):
        return self.fc2(self.fc1(x))

def get_mp_threshold_tail_asli(S, m, n, q=0.25):
    """[KODE ASLI - disimpan hanya untuk A/B] Estimasi noise dari rata-rata ekor 25%"""
    tail_len = max(1, int(len(S) * q))
    s_tail = S[-tail_len:]
    sigma_noise_sq = (s_tail**2).mean().item() / n
    Q = m / n
    lambda_max = sigma_noise_sq * ((1 + np.sqrt(1.0 / Q))**2) if Q > 0 else 0
    return np.sqrt(n * lambda_max) if lambda_max > 0 else 0

@lru_cache(maxsize=None)
def mp_quantile(beta, q):
    """[FIX 1a] Kuantil ke-q distribusi Marchenko-Pastur (sigma=1, rasio aspek beta <= 1)"""
    lower, upper = (1 - np.sqrt(beta)) ** 2, (1 + np.sqrt(beta)) ** 2
    x = np.linspace(lower, upper, 20001)[1:-1]
    pdf = np.sqrt((upper - x) * (x - lower)) / (2 * np.pi * beta * x)
    cdf = cumulative_trapezoid(pdf, x, initial=0)
    return float(np.interp(q, cdf / cdf[-1], x))

def get_mp_threshold_tail(S, m, n, q=0.25):
    """[FIX 1b] Sigma dari pencocokan KUANTIL ke-q spektrum dengan kuantil MP (bukan rata-rata ekor),
    lalu tepi bulk yang benar: sigma * (sqrt(m) + sqrt(n)) (rasio aspek tidak lagi terbalik)"""
    s = S.cpu().numpy()
    small, large = sorted((m, n))
    beta = min(small / large, 0.999)
    sigma = np.quantile(s, q) / np.sqrt(large * mp_quantile(beta, q))
    return sigma * (np.sqrt(small) + np.sqrt(large))

def compress_proposed(model, threshold_fn, mean_mode):
    comp_net = nn.Sequential()

    # [FIX 2] mean_mode="analitik": relu(mean) seperti kode asli; "batch": aktivasi NYATA dari batch kalibrasi.
    # Keduanya memakai rumus yang sama; yang berbeda hanya isi act_orig/act_comp (1 x n vs N x n).
    with torch.no_grad():
        if mean_mode == "batch":
            calibration = X_train_tensor[torch.randperm(len(X_train_tensor))[:1024]]
            act_orig, act_comp = calibration, calibration.clone()
        else:
            act_orig = X_train_tensor.mean(dim=0, keepdim=True)
            act_comp = act_orig.clone()
        
    layers = [model.fc1, model.fc2, model.fc3]
    ranks = []
    
    for idx, layer in enumerate(layers):
        W = layer.weight.data
        b = layer.bias.data
        m, n = W.shape
        
        U, S, Vh = torch.linalg.svd(W, full_matrices=False)
        thresh = threshold_fn(S, m, n, q=0.25)
        k = max(1, int((S > thresh).sum().item()))

        if k * (m + n) >= m * n:
            # [FIX 3] faktorisasi tidak menghemat parameter -> pertahankan layer asli
            module, W_rec, b_adj, k = copy.deepcopy(layer), W, b, min(m, n)
        else:
            U_k = U[:, :k]
            S_k = S[:k]
            Vh_k = Vh[:k, :]

            sqrt_S_k = torch.diag(torch.sqrt(S_k))
            W_A = sqrt_S_k @ Vh_k
            W_B = U_k @ sqrt_S_k
            W_rec = U_k @ torch.diag(S_k) @ Vh_k

            b_adj = b + W @ act_orig.mean(dim=0) - W_rec @ act_comp.mean(dim=0)
            module = FactoredLinear(W_A, W_B, b_adj)

        ranks.append((k, min(m, n)))
        comp_net.add_module(f"layer{idx+1}", module)
        comp_net.add_module(f"relu{idx+1}", nn.ReLU())
        
        with torch.no_grad():
            act_orig = torch.relu(act_orig @ W.T + b)
            act_comp = torch.relu(act_comp @ W_rec.T + b_adj)
            
    comp_net.add_module("head", copy.deepcopy(model.head))
    return comp_net, ranks

# ==========================================
# 4. PIPELINE EVALUASI & BENCHMARKING
# ==========================================
def get_metrics(net, x_tensor):
    net.eval()
    with torch.no_grad():
        logits = net(x_tensor).squeeze(1)    # [FIX 4] squeeze(1): aman untuk batch 1 sampel
        probs = torch.sigmoid(logits).cpu().numpy()
        preds = (probs >= 0.5).astype(int)
    return preds, probs

preds_base, probs_base = get_metrics(base_model, X_test_tensor)

acc_base = accuracy_score(y_test, preds_base)
f1_base = f1_score(y_test, preds_base)
auc_base = roc_auc_score(y_test, probs_base)
err_b = np.abs(y_test - probs_base)
p_base = sum(p.numel() for p in base_model.parameters())

batch_sizes = [1, 16, 64, 256, 1024, 4096, len(X_test_tensor)]

def bench_model(net, x_data, runs=100):
    net.eval()
    with torch.no_grad():
        for _ in range(15): _ = net(x_data) # Warmup
        timings = []
        for _ in range(runs):
            t0 = time.perf_counter()
            _ = net(x_data)
            t1 = time.perf_counter()
            timings.append((t1 - t0) * 1000)
    return np.mean(timings), np.std(timings)

print("\nMengukur latensi baseline...")
base_latency = {bs: bench_model(base_model, X_test_tensor[:bs]) for bs in batch_sizes}

def evaluate_and_print(variant_name, comp_model, rank_info):
    preds_comp, probs_comp = get_metrics(comp_model, X_test_tensor)

    acc_comp = accuracy_score(y_test, preds_comp)
    f1_comp = f1_score(y_test, preds_comp)
    auc_comp = roc_auc_score(y_test, probs_comp)

    b_mc = int(((preds_base == y_test) & (preds_comp != y_test)).sum())
    c_mc = int(((preds_base != y_test) & (preds_comp == y_test)).sum())
    p_mc = 1.0 if (b_mc + c_mc) == 0 else stats.binomtest(min(b_mc, c_mc), b_mc + c_mc, 0.5).pvalue

    err_c = np.abs(y_test - probs_comp)
    p_tt = 1.0 if np.allclose(err_b, err_c) else stats.ttest_rel(err_b, err_c).pvalue   # [FIX 5] hindari nan

    p_comp = sum(p.numel() for p in comp_model.parameters())

    latency_results = []
    for bs in batch_sizes:
        sample_data = X_test_tensor[:bs]

        mean_base, std_base = base_latency[bs]
        mean_comp, std_comp = bench_model(comp_model, sample_data)
        speedup = mean_base / mean_comp

        throughput_base = (bs / (mean_base / 1000))
        throughput_comp = (bs / (mean_comp / 1000))

        latency_results.append({
            "Batch Size": bs,
            "Lat Baseline (ms)": f"{mean_base:.4f} ± {std_base:.4f}",
            "Lat Proposed (ms)": f"{mean_comp:.4f} ± {std_comp:.4f}",
            "Speedup (x)": f"{speedup:.2f}x",
            "Throughput Base": f"{throughput_base:,.0f}",
            "Throughput Comp": f"{throughput_comp:,.0f}"
        })

    df_lat = pd.DataFrame(latency_results)

    # ==========================================
    # 5. CETAK HASIL AKHIR
    # ==========================================
    print("\n" + "="*80)
    print(f"=== {variant_name} ===")
    print("="*80)
    print(f"Rasio Rank Terpilih : L1={rank_info[0][0]}/{rank_info[0][1]}, L2={rank_info[1][0]}/{rank_info[1][1]}, L3={rank_info[2][0]}/{rank_info[2][1]}")
    print(f"Params Baseline     : {p_base:,}")
    print(f"Params Proposed     : {p_comp:,} (Hemat {((1 - p_comp/p_base)*100):.2f}%)")
    print("-" * 50)
    print(f"Akurasi             : Baseline {acc_base*100:.2f}% | Proposed {acc_comp*100:.2f}%")
    print(f"F1-Score            : Baseline {f1_base*100:.2f}% | Proposed {f1_comp*100:.2f}%")
    print(f"AUC-ROC             : Baseline {auc_base*100:.2f}% | Proposed {auc_comp*100:.2f}%")
    print(f"Pos-rate prediksi   : Baseline {preds_base.mean()*100:.2f}% | Proposed {preds_comp.mean()*100:.2f}% | Sebenarnya {y_test.mean()*100:.2f}%")
    print("-" * 50)
    print(f"McNemar p-value     : {p_mc:.4f} (p>0.05 = tidak terbukti berbeda, BUKAN bukti identik/ekuivalen)")
    print(f"Paired t-test p-val : {p_tt:.4f}")
    print("\n--- Latensi (single-thread) ---")
    print(df_lat.to_string(index=False))

# ==========================================
# 6. A/B: KODE ASLI vs PERBAIKAN (data yang memutuskan)
# ==========================================
VARIAN = {
    "A. KODE ASLI (ambang asli + relu(mean))": (get_mp_threshold_tail_asli, "analitik"),
    "B. Ambang MP diperbaiki saja": (get_mp_threshold_tail, "analitik"),
    "C. Ambang MP + kalibrasi batch (perbaikan penuh)": (get_mp_threshold_tail, "batch"),
}

print("\n[2/3] Mengeksekusi Kompresi: tiga varian (A/B/C)...")
for variant_name, (threshold_fn, mean_mode) in VARIAN.items():
    comp_model, rank_info = compress_proposed(base_model, threshold_fn, mean_mode)
    evaluate_and_print(variant_name, comp_model, rank_info)

In [ ]:
import time
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from functools import lru_cache
from itertools import chain
from scipy.integrate import cumulative_trapezoid
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 0. PENGATURAN REPRODUKSIBILITAS & CPU (PENTING UNTUK KAGGLE)
# ==========================================
SEEDS = [42, 0, 1, 2, 3]   # seed 42 pertama = mereproduksi run sebelumnya (split, init, dan latensi dari seed ini)

device = torch.device("cpu")
# Mengunci thread CPU ke-1 untuk latensi yang deterministik dan stabil
torch.set_num_threads(1) 

# ==========================================
# 1. PERSIAPAN DATASET REAL (ADULT INCOME)
# ==========================================
print("Mengunduh dataset Adult Income dari OpenML...")
X_raw, y_raw = fetch_openml(name='adult', version=2, return_X_y=True, as_frame=True)
df = pd.concat([X_raw, y_raw], axis=1).dropna()

X_df = df.iloc[:, :-1]
y_df = df.iloc[:, -1]

# Binerisasi target (>50K)
y_binary = (y_df.astype(str).str.contains('>50K')).astype(int).values
X_encoded = pd.get_dummies(X_df, drop_first=True).astype(float).values
input_dim = X_encoded.shape[1]
print(f"Dimensi Input: {input_dim}")

def prepare_data(seed):
    """Split dan scaler per seed (scaler hanya di-fit pada data train)"""
    X_train, X_test, y_train, y_test = train_test_split(
        X_encoded, y_binary, test_size=0.2, random_state=seed, stratify=y_binary
    )
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
    X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
    return X_train_tensor, y_train_tensor, X_test_tensor, y_test

# ==========================================
# 2. ARSITEKTUR BASELINE (WIDE MLP)
# ==========================================
class WideMLP(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, 1024)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(1024, 512)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(512, 256)
        self.relu3 = nn.ReLU()
        self.head = nn.Linear(256, 1)

    def forward(self, x):
        x = self.relu1(self.fc1(x))
        x = self.relu2(self.fc2(x))
        x = self.relu3(self.fc3(x))
        return self.head(x)

def train_baseline(X_train_tensor, y_train_tensor):
    base_model = WideMLP(input_dim)
    optimizer = optim.Adam(base_model.parameters(), lr=0.002)
    criterion = nn.BCEWithLogitsLoss()

    base_model.train()
    batch_size_train = 256
    n_samples = len(X_train_tensor)

    for epoch in range(15):
        permutation = torch.randperm(n_samples)
        for i in range(0, n_samples, batch_size_train):
            indices = permutation[i:i+batch_size_train]
            batch_x, batch_y = X_train_tensor[indices], y_train_tensor[indices]

            optimizer.zero_grad()
            loss = criterion(base_model(batch_x), batch_y)
            loss.backward()
            optimizer.step()

    base_model.eval()
    return base_model

# ==========================================
# 3. METODE USULAN: MP-TAIL (q=0.25) + SEQ BIAS
# ==========================================
class FactoredLinear(nn.Module):
    def __init__(self, W_A, W_B, bias_adj):
        super().__init__()
        k, in_f = W_A.shape
        out_f, _ = W_B.shape
        self.fc1 = nn.Linear(in_f, k, bias=False)
        self.fc2 = nn.Linear(k, out_f, bias=True)
        
        with torch.no_grad():
            self.fc1.weight.copy_(W_A)
            self.fc2.weight.copy_(W_B)
            self.fc2.bias.copy_(bias_adj)

    def forward(self, x):
        return self.fc2(self.fc1(x))

def get_mp_threshold_tail_asli(S, m, n, q=0.25):
    """[KODE ASLI - disimpan hanya untuk A/B] Estimasi noise dari rata-rata ekor 25%"""
    tail_len = max(1, int(len(S) * q))
    s_tail = S[-tail_len:]
    sigma_noise_sq = (s_tail**2).mean().item() / n
    Q = m / n
    lambda_max = sigma_noise_sq * ((1 + np.sqrt(1.0 / Q))**2) if Q > 0 else 0
    return np.sqrt(n * lambda_max) if lambda_max > 0 else 0

@lru_cache(maxsize=None)
def mp_quantile(beta, q):
    """Kuantil ke-q distribusi Marchenko-Pastur (sigma=1, rasio aspek beta <= 1)"""
    lower, upper = (1 - np.sqrt(beta)) ** 2, (1 + np.sqrt(beta)) ** 2
    x = np.linspace(lower, upper, 20001)[1:-1]
    pdf = np.sqrt((upper - x) * (x - lower)) / (2 * np.pi * beta * x)
    cdf = cumulative_trapezoid(pdf, x, initial=0)
    return float(np.interp(q, cdf / cdf[-1], x))

def get_mp_threshold_tail(S, m, n, q=0.25):
    """Sigma dari pencocokan KUANTIL ke-q spektrum dengan kuantil MP (bukan rata-rata ekor),
    lalu tepi bulk yang benar: sigma * (sqrt(m) + sqrt(n)) (rasio aspek tidak lagi terbalik)"""
    s = S.cpu().numpy()
    small, large = sorted((m, n))
    beta = min(small / large, 0.999)
    sigma = np.quantile(s, q) / np.sqrt(large * mp_quantile(beta, q))
    return sigma * (np.sqrt(small) + np.sqrt(large))

def compress_proposed(model, threshold_fn, mean_mode, X_train_tensor):
    comp_net = nn.Sequential()

    # mean_mode="analitik": relu(mean) seperti kode asli; "batch": aktivasi NYATA dari batch kalibrasi;
    # "none": tanpa kompensasi bias (ablation). act_orig/act_comp berukuran (1 x n) atau (N x n).
    with torch.no_grad():
        if mean_mode == "batch":
            calibration = X_train_tensor[torch.randperm(len(X_train_tensor))[:1024]]
            act_orig, act_comp = calibration, calibration.clone()
        else:
            act_orig = X_train_tensor.mean(dim=0, keepdim=True)
            act_comp = act_orig.clone()
        
    layers = [model.fc1, model.fc2, model.fc3]
    ranks = []
    
    for idx, layer in enumerate(layers):
        W = layer.weight.data
        b = layer.bias.data
        m, n = W.shape
        
        U, S, Vh = torch.linalg.svd(W, full_matrices=False)
        thresh = threshold_fn(S, m, n, q=0.25)
        k = max(1, int((S > thresh).sum().item()))

        if k * (m + n) >= m * n:
            # faktorisasi tidak menghemat parameter -> pertahankan layer asli
            module, W_rec, b_adj, k = copy.deepcopy(layer), W, b, min(m, n)
        else:
            U_k = U[:, :k]
            S_k = S[:k]
            Vh_k = Vh[:k, :]

            sqrt_S_k = torch.diag(torch.sqrt(S_k))
            W_A = sqrt_S_k @ Vh_k
            W_B = U_k @ sqrt_S_k
            W_rec = U_k @ torch.diag(S_k) @ Vh_k

            b_adj = b if mean_mode == "none" else b + W @ act_orig.mean(dim=0) - W_rec @ act_comp.mean(dim=0)
            module = FactoredLinear(W_A, W_B, b_adj)

        ranks.append((k, min(m, n)))
        comp_net.add_module(f"layer{idx+1}", module)
        comp_net.add_module(f"relu{idx+1}", nn.ReLU())
        
        with torch.no_grad():
            act_orig = torch.relu(act_orig @ W.T + b)
            act_comp = torch.relu(act_comp @ W_rec.T + b_adj)
            
    comp_net.add_module("head", copy.deepcopy(model.head))
    return comp_net, ranks

# ==========================================
# 4. PIPELINE EVALUASI (PER SEED)
# ==========================================
def get_metrics(net, x_tensor):
    net.eval()
    with torch.no_grad():
        logits = net(x_tensor).squeeze(1)
        probs = torch.sigmoid(logits).cpu().numpy()
        preds = (probs >= 0.5).astype(int)
    return preds, probs

def evaluate_variant(variant_name, net, rank_info, seed, X_test_tensor, y_test, reference):
    ref_preds, ref_probs, ref_params = reference
    preds, probs = get_metrics(net, X_test_tensor)

    acc, f1, auc = accuracy_score(y_test, preds), f1_score(y_test, preds), roc_auc_score(y_test, probs)
    acc_ref, f1_ref, auc_ref = accuracy_score(y_test, ref_preds), f1_score(y_test, ref_preds), roc_auc_score(y_test, ref_probs)

    b_mc = int(((ref_preds == y_test) & (preds != y_test)).sum())
    c_mc = int(((ref_preds != y_test) & (preds == y_test)).sum())
    p_mc = 1.0 if (b_mc + c_mc) == 0 else stats.binomtest(min(b_mc, c_mc), b_mc + c_mc, 0.5).pvalue

    err_ref, err = np.abs(y_test - ref_probs), np.abs(y_test - probs)
    p_tt = 1.0 if np.allclose(err_ref, err) else stats.ttest_rel(err_ref, err).pvalue

    params = sum(p.numel() for p in net.parameters())
    return {
        "variant": variant_name, "seed": seed, "params": params, "hemat": (1 - params / ref_params) * 100,
        "acc": acc * 100, "f1": f1 * 100, "auc": auc * 100, "pos_rate": preds.mean() * 100,
        "d_acc": (acc - acc_ref) * 100, "d_f1": (f1 - f1_ref) * 100, "d_auc": (auc - auc_ref) * 100,
        "p_mcnemar": p_mc, "p_ttest": p_tt,
        "ranks": "-" if rank_info is None else ", ".join(f"{k}/{full}" for k, full in rank_info),
    }

# ==========================================
# 5. A/B/C/D: KODE ASLI vs PERBAIKAN vs ABLATION TANPA BIAS
# ==========================================
VARIAN = {
    "A. KODE ASLI (ambang asli + relu(mean))": (get_mp_threshold_tail_asli, "analitik"),
    "B. Ambang MP diperbaiki saja": (get_mp_threshold_tail, "analitik"),
    "C. Ambang MP + kalibrasi batch": (get_mp_threshold_tail, "batch"),
    "D. Ambang MP, TANPA bias (ablation)": (get_mp_threshold_tail, "none"),
}

def run_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    X_train_tensor, y_train_tensor, X_test_tensor, y_test = prepare_data(seed)

    base_model = train_baseline(X_train_tensor, y_train_tensor)
    preds_base, probs_base = get_metrics(base_model, X_test_tensor)
    reference = (preds_base, probs_base, sum(p.numel() for p in base_model.parameters()))

    models, records = {}, [evaluate_variant("0. Baseline (Full)", base_model, None, seed, X_test_tensor, y_test, reference)]
    for variant_name, (threshold_fn, mean_mode) in VARIAN.items():
        comp_model, rank_info = compress_proposed(base_model, threshold_fn, mean_mode, X_train_tensor)
        models[variant_name] = comp_model
        records.append(evaluate_variant(variant_name, comp_model, rank_info, seed, X_test_tensor, y_test, reference))
    print(f"Seed {seed} selesai")
    return records, base_model, models, X_test_tensor

print(f"\n[1/3] Melatih baseline dan mengompresi untuk {len(SEEDS)} seed (15 epoch per seed)...")
results = [run_seed(seed) for seed in SEEDS]
records = list(chain.from_iterable(seed_records for seed_records, _, _, _ in results))

# ==========================================
# 6. RINGKASAN (mean ± std dari semua seed)
# ==========================================
summary = pd.DataFrame(records).groupby("variant", sort=False).agg(
    params=("params", "mean"), hemat_mean=("hemat", "mean"), hemat_std=("hemat", "std"),
    acc_mean=("acc", "mean"), acc_std=("acc", "std"), f1_mean=("f1", "mean"), f1_std=("f1", "std"),
    auc_mean=("auc", "mean"), auc_std=("auc", "std"), pos_rate=("pos_rate", "mean"),
    d_acc_mean=("d_acc", "mean"), d_acc_std=("d_acc", "std"), d_f1_mean=("d_f1", "mean"), d_f1_std=("d_f1", "std"),
    p_mcnemar=("p_mcnemar", "median"), p_ttest=("p_ttest", "median"), ranks=("ranks", "first"),
)
summary.to_csv("hasil_adult_5seed.csv")

def pm(mean_col, std_col, fmt="{:.2f}", sign=False):
    return summary[mean_col].map(("{:+.2f}" if sign else fmt).format) + " ± " + summary[std_col].map("{:.2f}".format)

table = pd.DataFrame({
    "Params": summary["params"].round().astype(int),
    "Hemat (%)": pm("hemat_mean", "hemat_std", "{:.1f}"),
    f"Rank (seed {SEEDS[0]})": summary["ranks"],
    "Akurasi (%)": pm("acc_mean", "acc_std"),
    "F1 (%)": pm("f1_mean", "f1_std"),
    "AUC (%)": pm("auc_mean", "auc_std"),
    "dAkurasi (pp)": pm("d_acc_mean", "d_acc_std", sign=True),
    "dF1 (pp)": pm("d_f1_mean", "d_f1_std", sign=True),
    "Pos-rate (%)": summary["pos_rate"].map("{:.2f}".format),
    "McNemar p": summary["p_mcnemar"].map("{:.4f}".format),
    "t-test p": summary["p_ttest"].map("{:.4f}".format),
})

pd.set_option("display.width", 320)
pd.set_option("display.max_columns", 50)
print("\n" + "=" * 150)
print(f"=== AKURASI & PROFIL PARAMETER (mean ± std dari {len(SEEDS)} seed/split; p-value = median antar seed) ===")
print("=" * 150)
print(table.to_string())
print("Catatan: p>0.05 = tidak terbukti berbeda, BUKAN bukti identik/ekuivalen. Pos-rate sebenarnya ~24.8%.")

# ==========================================
# 7. BENCHMARK LATENSI (SEED PERTAMA, SINGLE-THREAD)
# ==========================================
base_model_0, models_0, X_test_0 = results[0][1], results[0][2], results[0][3]
batch_sizes = [1, 16, 64, 256, 1024, 4096, len(X_test_0)]

def bench_model(net, x_data, runs=100):
    net.eval()
    with torch.no_grad():
        for _ in range(15): _ = net(x_data) # Warmup
        timings = []
        for _ in range(runs):
            t0 = time.perf_counter()
            _ = net(x_data)
            t1 = time.perf_counter()
            timings.append((t1 - t0) * 1000)
    return np.mean(timings), np.std(timings)

def latency_table(comp_model):
    latency_results = []
    for bs in batch_sizes:
        sample_data = X_test_0[:bs]
        mean_base, std_base = base_latency[bs]
        mean_comp, std_comp = bench_model(comp_model, sample_data)
        latency_results.append({
            "Batch Size": bs,
            "Lat Baseline (ms)": f"{mean_base:.4f} ± {std_base:.4f}",
            "Lat Proposed (ms)": f"{mean_comp:.4f} ± {std_comp:.4f}",
            "Speedup (x)": f"{mean_base / mean_comp:.2f}x",
            "Throughput Base": f"{bs / (mean_base / 1000):,.0f}",
            "Throughput Comp": f"{bs / (mean_comp / 1000):,.0f}",
        })
    return pd.DataFrame(latency_results)

print("\nMengukur latensi baseline...")
base_latency = {bs: bench_model(base_model_0, X_test_0[:bs]) for bs in batch_sizes}

for variant_name, comp_model in models_0.items():
    print("\n" + "=" * 80)
    print(f"=== LATENSI: {variant_name} (seed {SEEDS[0]}) ===")
    print("=" * 80)
    print(latency_table(comp_model).to_string(index=False))

# CHECK

In [1]:
import time
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from functools import lru_cache
from itertools import chain
from scipy.integrate import cumulative_trapezoid
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 0. PENGATURAN REPRODUKSIBILITAS & CPU (PENTING UNTUK KAGGLE)
# ==========================================
SEEDS = [42, 0, 1, 2, 3]   # seed 42 pertama = mereproduksi run sebelumnya (split, init, dan latensi dari seed ini)

device = torch.device("cpu")
# Mengunci thread CPU ke-1 untuk latensi yang deterministik dan stabil
torch.set_num_threads(1) 

# ==========================================
# 1. PERSIAPAN DATASET REAL (ADULT INCOME)
# ==========================================
print("Mengunduh dataset Adult Income dari OpenML...")
X_raw, y_raw = fetch_openml(name='adult', version=2, return_X_y=True, as_frame=True)
df = pd.concat([X_raw, y_raw], axis=1).dropna()

X_df = df.iloc[:, :-1]
y_df = df.iloc[:, -1]

# Binerisasi target (>50K)
y_binary = (y_df.astype(str).str.contains('>50K')).astype(int).values
X_encoded = pd.get_dummies(X_df, drop_first=True).astype(float).values
input_dim = X_encoded.shape[1]
print(f"Dimensi Input: {input_dim}")

def prepare_data(seed):
    """Split dan scaler per seed (scaler hanya di-fit pada data train)"""
    X_train, X_test, y_train, y_test = train_test_split(
        X_encoded, y_binary, test_size=0.2, random_state=seed, stratify=y_binary
    )
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
    X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
    return X_train_tensor, y_train_tensor, X_test_tensor, y_test

# ==========================================
# 2. ARSITEKTUR BASELINE (WIDE MLP)
# ==========================================
class WideMLP(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, 1024)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(1024, 512)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(512, 256)
        self.relu3 = nn.ReLU()
        self.head = nn.Linear(256, 1)

    def forward(self, x):
        x = self.relu1(self.fc1(x))
        x = self.relu2(self.fc2(x))
        x = self.relu3(self.fc3(x))
        return self.head(x)

def train_baseline(X_train_tensor, y_train_tensor):
    base_model = WideMLP(input_dim)
    optimizer = optim.Adam(base_model.parameters(), lr=0.002)
    criterion = nn.BCEWithLogitsLoss()

    base_model.train()
    batch_size_train = 256
    n_samples = len(X_train_tensor)

    for epoch in range(15):
        permutation = torch.randperm(n_samples)
        for i in range(0, n_samples, batch_size_train):
            indices = permutation[i:i+batch_size_train]
            batch_x, batch_y = X_train_tensor[indices], y_train_tensor[indices]

            optimizer.zero_grad()
            loss = criterion(base_model(batch_x), batch_y)
            loss.backward()
            optimizer.step()

    base_model.eval()
    return base_model

# ==========================================
# 3. METODE USULAN: SVD DENGAN AMBANG MARCHENKO-PASTUR (TAIL q=0.25), TANPA KOMPENSASI BIAS
# ==========================================
class FactoredLinear(nn.Module):
    def __init__(self, W_A, W_B, bias):
        super().__init__()
        k, in_f = W_A.shape
        out_f, _ = W_B.shape
        self.fc1 = nn.Linear(in_f, k, bias=False)
        self.fc2 = nn.Linear(k, out_f, bias=True)
        
        with torch.no_grad():
            self.fc1.weight.copy_(W_A)
            self.fc2.weight.copy_(W_B)
            self.fc2.bias.copy_(bias)

    def forward(self, x):
        return self.fc2(self.fc1(x))

@lru_cache(maxsize=None)
def mp_quantile(beta, q):
    """Kuantil ke-q distribusi Marchenko-Pastur (sigma=1, rasio aspek beta <= 1)"""
    lower, upper = (1 - np.sqrt(beta)) ** 2, (1 + np.sqrt(beta)) ** 2
    x = np.linspace(lower, upper, 20001)[1:-1]
    pdf = np.sqrt((upper - x) * (x - lower)) / (2 * np.pi * beta * x)
    cdf = cumulative_trapezoid(pdf, x, initial=0)
    return float(np.interp(q, cdf / cdf[-1], x))

def get_mp_threshold_tail(S, m, n, q=0.25):
    """[PROPOSED] Sigma dari pencocokan KUANTIL ke-q spektrum dengan kuantil MP (bukan rata-rata ekor),
    lalu tepi bulk yang benar: sigma * (sqrt(m) + sqrt(n))"""
    s = S.cpu().numpy()
    small, large = sorted((m, n))
    beta = min(small / large, 0.999)
    sigma = np.quantile(s, q) / np.sqrt(large * mp_quantile(beta, q))
    return sigma * (np.sqrt(small) + np.sqrt(large))

def get_mp_threshold_tail_asli(S, m, n, q=0.25):
    """[ABLATION] Ambang kode awal: estimasi noise dari rata-rata ekor 25% dan rasio aspek terbalik"""
    tail_len = max(1, int(len(S) * q))
    s_tail = S[-tail_len:]
    sigma_noise_sq = (s_tail**2).mean().item() / n
    Q = m / n
    lambda_max = sigma_noise_sq * ((1 + np.sqrt(1.0 / Q))**2) if Q > 0 else 0
    return np.sqrt(n * lambda_max) if lambda_max > 0 else 0

def compress_proposed(model, threshold_fn):
    comp_net = nn.Sequential()
    layers = [model.fc1, model.fc2, model.fc3]
    ranks = []
    
    for idx, layer in enumerate(layers):
        W = layer.weight.data
        b = layer.bias.data
        m, n = W.shape
        
        U, S, Vh = torch.linalg.svd(W, full_matrices=False)
        thresh = threshold_fn(S, m, n, q=0.25)
        k = max(1, int((S > thresh).sum().item()))

        if k * (m + n) >= m * n:
            # faktorisasi tidak menghemat parameter -> pertahankan layer asli
            module, k = copy.deepcopy(layer), min(m, n)
        else:
            sqrt_S_k = torch.diag(torch.sqrt(S[:k]))
            module = FactoredLinear(sqrt_S_k @ Vh[:k, :], U[:, :k] @ sqrt_S_k, b)

        ranks.append((k, min(m, n)))
        comp_net.add_module(f"layer{idx+1}", module)
        comp_net.add_module(f"relu{idx+1}", nn.ReLU())
            
    comp_net.add_module("head", copy.deepcopy(model.head))
    return comp_net, ranks

# ==========================================
# 4. PIPELINE EVALUASI (PER SEED)
# ==========================================
def get_metrics(net, x_tensor):
    net.eval()
    with torch.no_grad():
        logits = net(x_tensor).squeeze(1)
        probs = torch.sigmoid(logits).cpu().numpy()
        preds = (probs >= 0.5).astype(int)
    return preds, probs

def evaluate_variant(variant_name, net, rank_info, seed, X_test_tensor, y_test, reference):
    ref_preds, ref_probs, ref_params = reference
    preds, probs = get_metrics(net, X_test_tensor)

    acc, f1, auc = accuracy_score(y_test, preds), f1_score(y_test, preds), roc_auc_score(y_test, probs)
    acc_ref, f1_ref, auc_ref = accuracy_score(y_test, ref_preds), f1_score(y_test, ref_preds), roc_auc_score(y_test, ref_probs)

    b_mc = int(((ref_preds == y_test) & (preds != y_test)).sum())
    c_mc = int(((ref_preds != y_test) & (preds == y_test)).sum())
    p_mc = 1.0 if (b_mc + c_mc) == 0 else stats.binomtest(min(b_mc, c_mc), b_mc + c_mc, 0.5).pvalue

    err_ref, err = np.abs(y_test - ref_probs), np.abs(y_test - probs)
    p_tt = 1.0 if np.allclose(err_ref, err) else stats.ttest_rel(err_ref, err).pvalue

    params = sum(p.numel() for p in net.parameters())
    return {
        "variant": variant_name, "seed": seed, "params": params, "hemat": (1 - params / ref_params) * 100,
        "acc": acc * 100, "f1": f1 * 100, "auc": auc * 100, "pos_rate": preds.mean() * 100,
        "d_acc": (acc - acc_ref) * 100, "d_f1": (f1 - f1_ref) * 100, "d_auc": (auc - auc_ref) * 100,
        "p_mcnemar": p_mc, "p_ttest": p_tt,
        "ranks": "-" if rank_info is None else ", ".join(f"{k}/{full}" for k, full in rank_info),
    }

# ==========================================
# 5. PROPOSED vs ABLATION AMBANG
# ==========================================
VARIAN = {
    "1. PROPOSED: SVD + ambang MP (kuantil q=0.25)": get_mp_threshold_tail,
    "2. Ablation: ambang awal (rata-rata ekor)": get_mp_threshold_tail_asli,
}

def run_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    X_train_tensor, y_train_tensor, X_test_tensor, y_test = prepare_data(seed)

    base_model = train_baseline(X_train_tensor, y_train_tensor)
    preds_base, probs_base = get_metrics(base_model, X_test_tensor)
    reference = (preds_base, probs_base, sum(p.numel() for p in base_model.parameters()))

    models, records = {}, [evaluate_variant("0. Baseline (Full)", base_model, None, seed, X_test_tensor, y_test, reference)]
    for variant_name, threshold_fn in VARIAN.items():
        comp_model, rank_info = compress_proposed(base_model, threshold_fn)
        models[variant_name] = comp_model
        records.append(evaluate_variant(variant_name, comp_model, rank_info, seed, X_test_tensor, y_test, reference))
    print(f"Seed {seed} selesai")
    return records, base_model, models, X_test_tensor

print(f"\n[1/3] Melatih baseline dan mengompresi untuk {len(SEEDS)} seed (15 epoch per seed)...")
results = [run_seed(seed) for seed in SEEDS]
records = list(chain.from_iterable(seed_records for seed_records, _, _, _ in results))

# ==========================================
# 6. RINGKASAN (mean ± std dan CI95 dari semua seed)
# ==========================================
summary = pd.DataFrame(records).groupby("variant", sort=False).agg(
    params=("params", "mean"), hemat_mean=("hemat", "mean"), hemat_std=("hemat", "std"),
    acc_mean=("acc", "mean"), acc_std=("acc", "std"), f1_mean=("f1", "mean"), f1_std=("f1", "std"),
    auc_mean=("auc", "mean"), auc_std=("auc", "std"), pos_rate=("pos_rate", "mean"),
    d_acc_mean=("d_acc", "mean"), d_acc_std=("d_acc", "std"), d_f1_mean=("d_f1", "mean"), d_f1_std=("d_f1", "std"),
    p_mcnemar=("p_mcnemar", "median"), p_ttest=("p_ttest", "median"), ranks=("ranks", "first"),
)
summary.to_csv("hasil_adult_proposed_mp.csv")

t_crit = stats.t.ppf(0.975, len(SEEDS) - 1)

def pm(mean_col, std_col, fmt="{:.2f}", sign=False):
    return summary[mean_col].map(("{:+.2f}" if sign else fmt).format) + " ± " + summary[std_col].map("{:.2f}".format)

def ci95(mean_col, std_col):
    half = t_crit * summary[std_col] / np.sqrt(len(SEEDS))
    return "[" + (summary[mean_col] - half).map("{:+.2f}".format) + ", " + (summary[mean_col] + half).map("{:+.2f}".format) + "]"

table = pd.DataFrame({
    "Params": summary["params"].round().astype(int),
    "Hemat (%)": pm("hemat_mean", "hemat_std", "{:.1f}"),
    f"Rank (seed {SEEDS[0]})": summary["ranks"],
    "Akurasi (%)": pm("acc_mean", "acc_std"),
    "F1 (%)": pm("f1_mean", "f1_std"),
    "AUC (%)": pm("auc_mean", "auc_std"),
    "dAkurasi (pp)": pm("d_acc_mean", "d_acc_std", sign=True),
    "CI95 dAkurasi": ci95("d_acc_mean", "d_acc_std"),
    "dF1 (pp)": pm("d_f1_mean", "d_f1_std", sign=True),
    "CI95 dF1": ci95("d_f1_mean", "d_f1_std"),
    "Pos-rate (%)": summary["pos_rate"].map("{:.2f}".format),
    "McNemar p": summary["p_mcnemar"].map("{:.4f}".format),
    "t-test p": summary["p_ttest"].map("{:.4f}".format),
})

pd.set_option("display.width", 320)
pd.set_option("display.max_columns", 50)
print("\n" + "=" * 170)
print(f"=== AKURASI & PROFIL PARAMETER (mean ± std dari {len(SEEDS)} seed/split; p-value = median antar seed) ===")
print("=" * 170)
print(table.to_string())
print("Catatan: CI95 dari std antar seed (split saling tumpang tindih, sedikit terlalu optimis). "
      "p>0.05 = tidak terbukti berbeda, BUKAN bukti ekuivalen. Pos-rate sebenarnya ~24.8%.")

# ==========================================
# 7. BENCHMARK LATENSI (SEED PERTAMA, SINGLE-THREAD)
# ==========================================
base_model_0, models_0, X_test_0 = results[0][1], results[0][2], results[0][3]
batch_sizes = [1, 16, 64, 256, 1024, 4096, len(X_test_0)]

def bench_model(net, x_data, runs=100):
    net.eval()
    with torch.no_grad():
        for _ in range(15): _ = net(x_data) # Warmup
        timings = []
        for _ in range(runs):
            t0 = time.perf_counter()
            _ = net(x_data)
            t1 = time.perf_counter()
            timings.append((t1 - t0) * 1000)
    return np.mean(timings), np.std(timings)

def latency_table(comp_model):
    latency_results = []
    for bs in batch_sizes:
        sample_data = X_test_0[:bs]
        mean_base, std_base = base_latency[bs]
        mean_comp, std_comp = bench_model(comp_model, sample_data)
        latency_results.append({
            "Batch Size": bs,
            "Lat Baseline (ms)": f"{mean_base:.4f} ± {std_base:.4f}",
            "Lat Proposed (ms)": f"{mean_comp:.4f} ± {std_comp:.4f}",
            "Speedup (x)": f"{mean_base / mean_comp:.2f}x",
            "Throughput Base": f"{bs / (mean_base / 1000):,.0f}",
            "Throughput Comp": f"{bs / (mean_comp / 1000):,.0f}",
        })
    return pd.DataFrame(latency_results)

print("\nMengukur latensi baseline...")
base_latency = {bs: bench_model(base_model_0, X_test_0[:bs]) for bs in batch_sizes}

for variant_name, comp_model in models_0.items():
    print("\n" + "=" * 80)
    print(f"=== LATENSI: {variant_name} (seed {SEEDS[0]}) ===")
    print("=" * 80)
    print(latency_table(comp_model).to_string(index=False))

Mengunduh dataset Adult Income dari OpenML...
Dimensi Input: 97

[1/3] Melatih baseline dan mengompresi untuk 5 seed (15 epoch per seed)...
Seed 42 selesai
Seed 0 selesai
Seed 1 selesai
Seed 2 selesai
Seed 3 selesai

=== AKURASI & PROFIL PARAMETER (mean ± std dari 5 seed/split; p-value = median antar seed) ===
                                               Params    Hemat (%)          Rank (seed 42)   Akurasi (%)        F1 (%)       AUC (%) dAkurasi (pp)   CI95 dAkurasi      dF1 (pp)        CI95 dF1 Pos-rate (%) McNemar p t-test p
variant                                                                                                                                                                                                                         
0. Baseline (Full)                             756737   0.0 ± 0.00                       -  84.85 ± 0.35  66.78 ± 1.04  90.18 ± 0.34  +0.00 ± 0.00  [+0.00, +0.00]  +0.00 ± 0.00  [+0.00, +0.00]        20.82    1.0000   1.0000
1. PROPOSED: 

# ABLATION

In [2]:
import time
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from functools import lru_cache
from itertools import chain, product
from scipy.integrate import cumulative_trapezoid
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (f1_score, accuracy_score, roc_auc_score, average_precision_score,
                             log_loss, precision_recall_curve)
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 0. KONFIGURASI EKSPERIMEN
# ==========================================
SEEDS = [42, 0, 1, 2, 3]            # seed 42 pertama; latensi diukur dari seed ini
EQUIV_MARGIN = 0.5                  # margin ekuivalensi akurasi (poin persentase). TETAPKAN SEBELUM MELIHAT HASIL
THRESHOLD_SWEEP = [0.5, 0.75, 1.5, 2.0, 3.0]   # pengali ambang MP untuk kurva trade-off (Pareto)
LAYER_NAMES = ["fc1", "fc2", "fc3"]

device = torch.device("cpu")
# Mengunci thread CPU ke-1 untuk latensi yang deterministik dan stabil
torch.set_num_threads(1)

# ==========================================
# 1. PERSIAPAN DATASET REAL (ADULT INCOME)
# ==========================================
print("Mengunduh dataset Adult Income dari OpenML...")
X_raw, y_raw = fetch_openml(name='adult', version=2, return_X_y=True, as_frame=True)
df = pd.concat([X_raw, y_raw], axis=1).dropna()

X_df = df.iloc[:, :-1]
y_df = df.iloc[:, -1]

# Binerisasi target (>50K)
y_binary = (y_df.astype(str).str.contains('>50K')).astype(int).values
X_encoded = pd.get_dummies(X_df, drop_first=True).astype(float).values
input_dim = X_encoded.shape[1]
print(f"Dimensi Input: {input_dim} | Total data: {len(X_encoded):,} | Proporsi >50K: {y_binary.mean():.3f}")

def prepare_data(seed):
    """Split train/val/test per seed; val (10% dari train) hanya untuk menala ambang keputusan F1;
    scaler hanya di-fit pada data train"""
    X_trainval, X_test, y_trainval, y_test = train_test_split(
        X_encoded, y_binary, test_size=0.2, random_state=seed, stratify=y_binary
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_trainval, y_trainval, test_size=0.1, random_state=seed, stratify=y_trainval
    )
    scaler = StandardScaler().fit(X_train)

    return {
        "X_train": torch.tensor(scaler.transform(X_train), dtype=torch.float32),
        "y_train": torch.tensor(y_train, dtype=torch.float32).unsqueeze(1),
        "X_val": torch.tensor(scaler.transform(X_val), dtype=torch.float32), "y_val": y_val,
        "X_test": torch.tensor(scaler.transform(X_test), dtype=torch.float32), "y_test": y_test,
    }

# ==========================================
# 2. ARSITEKTUR BASELINE (WIDE MLP)
# ==========================================
class WideMLP(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, 1024)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(1024, 512)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(512, 256)
        self.relu3 = nn.ReLU()
        self.head = nn.Linear(256, 1)

    def forward(self, x):
        x = self.relu1(self.fc1(x))
        x = self.relu2(self.fc2(x))
        x = self.relu3(self.fc3(x))
        return self.head(x)

def train_baseline(data):
    base_model = WideMLP(input_dim)
    optimizer = optim.Adam(base_model.parameters(), lr=0.002)
    criterion = nn.BCEWithLogitsLoss()

    base_model.train()
    batch_size_train = 256
    n_samples = len(data["X_train"])

    for epoch in range(15):
        permutation = torch.randperm(n_samples)
        for i in range(0, n_samples, batch_size_train):
            indices = permutation[i:i+batch_size_train]
            batch_x, batch_y = data["X_train"][indices], data["y_train"][indices]

            optimizer.zero_grad()
            loss = criterion(base_model(batch_x), batch_y)
            loss.backward()
            optimizer.step()

    base_model.eval()
    return base_model

# ==========================================
# 3. AMBANG / PEMILIHAN RANK
# (semua fungsi menerima spektrum singular value berupa array numpy, menurun)
# ==========================================
@lru_cache(maxsize=None)
def mp_quantile(beta, q):
    """Kuantil ke-q distribusi Marchenko-Pastur (sigma=1, rasio aspek beta <= 1)"""
    lower, upper = (1 - np.sqrt(beta)) ** 2, (1 + np.sqrt(beta)) ** 2
    x = np.linspace(lower, upper, 20001)[1:-1]
    pdf = np.sqrt((upper - x) * (x - lower)) / (2 * np.pi * beta * x)
    cdf = cumulative_trapezoid(pdf, x, initial=0)
    return float(np.interp(q, cdf / cdf[-1], x))

def get_mp_threshold_tail(S, m, n, q=0.25):
    """[PROPOSED] Sigma dari pencocokan KUANTIL ke-q spektrum dengan kuantil MP (bukan rata-rata ekor),
    lalu tepi bulk yang benar: sigma * (sqrt(m) + sqrt(n))"""
    small, large = sorted((m, n))
    beta = min(small / large, 0.999)
    sigma = np.quantile(S, q) / np.sqrt(large * mp_quantile(beta, q))
    return sigma * (np.sqrt(small) + np.sqrt(large))

def get_mp_threshold_tail_asli(S, m, n, q=0.25):
    """[ABLATION] Ambang kode awal: estimasi noise dari rata-rata ekor 25% dan rasio aspek terbalik"""
    tail_len = max(1, int(len(S) * q))
    s_tail = S[-tail_len:]
    sigma_noise_sq = float((s_tail ** 2).mean()) / n
    Q = m / n
    lambda_max = sigma_noise_sq * ((1 + np.sqrt(1.0 / Q)) ** 2) if Q > 0 else 0
    return np.sqrt(n * lambda_max) if lambda_max > 0 else 0

def gavish_donoho_threshold(S, m, n):
    """[PEMBANDING] Optimal hard threshold Gavish-Donoho (2014) untuk noise tak diketahui"""
    small, large = sorted((m, n))
    beta = small / large
    return (0.56 * beta**3 - 0.95 * beta**2 + 1.82 * beta + 1.43) * np.median(S)

def get_spectra(model):
    layers = [getattr(model, name) for name in LAYER_NAMES]
    shapes = [tuple(layer.weight.shape) for layer in layers]
    spectra = [torch.linalg.svdvals(layer.weight.data).numpy() for layer in layers]
    return spectra, shapes

def ranks_by_threshold(spectra, shapes, threshold_fn, multiplier=1.0):
    return [max(1, int((S > multiplier * threshold_fn(S, m, n)).sum())) for S, (m, n) in zip(spectra, shapes)]

def ranks_evr(spectra, evr):
    return [min(len(S), int((np.cumsum(S**2) / (S**2).sum() < evr).sum()) + 1) for S in spectra]

def ranks_uniform(shapes, ratio):
    return [max(1, int(round(ratio * min(m, n)))) for m, n in shapes]

# ==========================================
# 4. KOMPRESI: SVD (RANK DIBERIKAN) DAN MAGNITUDE PRUNING
# ==========================================
class FactoredLinear(nn.Module):
    def __init__(self, W_A, W_B, bias):
        super().__init__()
        k, in_f = W_A.shape
        out_f, _ = W_B.shape
        self.fc1 = nn.Linear(in_f, k, bias=False)
        self.fc2 = nn.Linear(k, out_f, bias=True)

        with torch.no_grad():
            self.fc1.weight.copy_(W_A)
            self.fc2.weight.copy_(W_B)
            self.fc2.bias.copy_(bias)

    def forward(self, x):
        return self.fc2(self.fc1(x))

def layer_params(m, n, k):
    # faktorisasi hanya dipakai jika menghemat parameter; selain itu layer asli dipertahankan
    return m * n + m if k * (m + n) >= m * n else k * (n + m) + m

def total_params(shapes, ranks, fixed_params):
    return sum(layer_params(m, n, k) for (m, n), k in zip(shapes, ranks)) + fixed_params

def effective_ranks_text(shapes, ranks):
    return ", ".join(f"{min(m, n) if k * (m + n) >= m * n else k}/{min(m, n)}" for (m, n), k in zip(shapes, ranks))

def compress_with_ranks(model, ranks):
    comp_net = nn.Sequential()
    for idx, (name, k) in enumerate(zip(LAYER_NAMES, ranks)):
        layer = getattr(model, name)
        W, b = layer.weight.data, layer.bias.data
        m, n = W.shape

        if k * (m + n) >= m * n:
            module = copy.deepcopy(layer)
        else:
            U, S, Vh = torch.linalg.svd(W, full_matrices=False)
            sqrt_S_k = torch.diag(torch.sqrt(S[:k]))
            module = FactoredLinear(sqrt_S_k @ Vh[:k, :], U[:, :k] @ sqrt_S_k, b)

        comp_net.add_module(f"layer{idx+1}", module)
        comp_net.add_module(f"relu{idx+1}", nn.ReLU())

    comp_net.add_module("head", copy.deepcopy(model.head))
    comp_net.eval()
    return comp_net

def match_budget(rank_fn, target_params, shapes, fixed_params, iterations=30):
    # Parameter total monoton naik terhadap pengatur rank (0..1): bisection mencari nilai terbesar
    # yang TIDAK melebihi anggaran parameter Proposed
    low, high = 0.0, 1.0
    for _ in range(iterations):
        mid = (low + high) / 2
        if total_params(shapes, rank_fn(mid), fixed_params) > target_params:
            high = mid
        else:
            low = mid
    return low

def magnitude_prune(model, ratio):
    model_pruned = copy.deepcopy(model)
    with torch.no_grad():
        for name in LAYER_NAMES:
            weight = getattr(model_pruned, name).weight
            weight.mul_((weight.abs() > torch.quantile(weight.abs().flatten(), ratio)).float())
    model_pruned.eval()
    return model_pruned

def prune_ratio_for_budget(model, target_params):
    weights_total = sum(getattr(model, name).weight.numel() for name in LAYER_NAMES)
    other_params = sum(p.numel() for p in model.parameters()) - weights_total
    return float(np.clip(1 - (target_params - other_params) / weights_total, 0.0, 0.999))

# ==========================================
# 5. EVALUASI: KUALITAS, EFISIENSI, UJI STATISTIK
# ==========================================
def get_metrics(net, x_tensor):
    net.eval()
    with torch.no_grad():
        probs = torch.sigmoid(net(x_tensor).squeeze(1)).cpu().numpy()
    return (probs >= 0.5).astype(int), probs

def count_effective_params(net):
    return sum(int((p != 0).sum().item()) for p in net.parameters())

def count_effective_flops(net):
    # FLOPs per sampel = 2 x bobot non-nol pada semua layer Linear
    return 2 * sum(int((m.weight != 0).sum().item()) for m in net.modules() if isinstance(m, nn.Linear))

def tuned_threshold(y_val, probs_val):
    precision, recall, thresholds = precision_recall_curve(y_val, probs_val)
    f1_curve = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
    return thresholds[np.argmax(f1_curve)]

def evaluate_variant(variant_name, net, ranks_text, seed, data, reference):
    ref_preds, ref_probs, ref_params = reference
    y_test = data["y_test"]
    preds, probs = get_metrics(net, data["X_test"])
    probs_val = get_metrics(net, data["X_val"])[1]
    threshold = tuned_threshold(data["y_val"], probs_val)       # ambang dituning di validation set, per model

    acc, acc_ref = accuracy_score(y_test, preds), accuracy_score(y_test, ref_preds)
    f1, f1_ref = f1_score(y_test, preds), f1_score(y_test, ref_preds)

    b_mc = int(((ref_preds == y_test) & (preds != y_test)).sum())
    c_mc = int(((ref_preds != y_test) & (preds == y_test)).sum())
    p_mc = 1.0 if (b_mc + c_mc) == 0 else stats.binomtest(min(b_mc, c_mc), b_mc + c_mc, 0.5).pvalue

    err_ref, err = np.abs(y_test - ref_probs), np.abs(y_test - probs)
    p_tt = 1.0 if np.allclose(err_ref, err) else stats.ttest_rel(err_ref, err).pvalue

    params = count_effective_params(net)
    return {
        "variant": variant_name, "seed": seed, "params": params, "hemat": (1 - params / ref_params) * 100,
        "flops": count_effective_flops(net), "acc": acc * 100, "f1": f1 * 100,
        "f1_tuned": f1_score(y_test, (probs >= threshold).astype(int)) * 100,
        "auc": roc_auc_score(y_test, probs) * 100, "pr_auc": average_precision_score(y_test, probs) * 100,
        "log_loss": log_loss(y_test, np.clip(probs, 1e-7, 1 - 1e-7)), "pos_rate": preds.mean() * 100,
        "d_acc": (acc - acc_ref) * 100, "d_f1": (f1 - f1_ref) * 100,
        "p_mcnemar": p_mc, "p_ttest": p_tt, "ranks": ranks_text,
    }

# ==========================================
# 6. SATU EKSPERIMEN LENGKAP (SATU SEED)
# ==========================================
PROPOSED = "01. PROPOSED: SVD + ambang MP (kuantil q=0.25)"

def run_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    data = prepare_data(seed)

    base_model = train_baseline(data)
    preds_base, probs_base = get_metrics(base_model, data["X_test"])
    reference = (preds_base, probs_base, count_effective_params(base_model))

    spectra, shapes = get_spectra(base_model)
    fixed_params = sum(p.numel() for p in base_model.head.parameters())

    # Anggaran parameter Proposed menjadi acuan untuk pembanding "anggaran sama"
    ranks_proposed = ranks_by_threshold(spectra, shapes, get_mp_threshold_tail)
    target_params = total_params(shapes, ranks_proposed, fixed_params)
    evr_matched = match_budget(lambda v: ranks_evr(spectra, v), target_params, shapes, fixed_params)
    ratio_matched = match_budget(lambda v: ranks_uniform(shapes, v), target_params, shapes, fixed_params)

    rank_plan = {
        PROPOSED: ranks_proposed,
        "02. Ablation: ambang awal (rata-rata ekor)": ranks_by_threshold(spectra, shapes, get_mp_threshold_tail_asli),
        "03. Gavish-Donoho (anggaran sendiri)": ranks_by_threshold(spectra, shapes, gavish_donoho_threshold),
        "04. EVR 85% (heuristik standar)": ranks_evr(spectra, 0.85),
        "05. EVR dicocokkan ke anggaran Proposed": ranks_evr(spectra, evr_matched),
        "06. Rank seragam dicocokkan ke anggaran Proposed": ranks_uniform(shapes, ratio_matched),
        **{f"{8 + i:02d}. Proposed, ambang x{mult}": ranks_by_threshold(spectra, shapes, get_mp_threshold_tail, mult)
           for i, mult in enumerate(THRESHOLD_SWEEP)},
    }
    models = {label: compress_with_ranks(base_model, ranks) for label, ranks in rank_plan.items()}
    models["07. Magnitude pruning dicocokkan ke anggaran Proposed"] = magnitude_prune(
        base_model, prune_ratio_for_budget(base_model, target_params))
    models = dict(sorted(models.items()))

    ranks_text = {label: effective_ranks_text(shapes, ranks) for label, ranks in rank_plan.items()}
    records = [evaluate_variant("00. Baseline (Full)", base_model, "-", seed, data, reference)]
    records += [evaluate_variant(label, net, ranks_text.get(label, "-"), seed, data, reference) for label, net in models.items()]
    print(f"Seed {seed} selesai")
    return records, base_model, models, data["X_test"]

print(f"\n[1/3] Melatih baseline dan mengompresi untuk {len(SEEDS)} seed (15 epoch per seed, semua metode zero-shot)...")
results = [run_seed(seed) for seed in SEEDS]
records = list(chain.from_iterable(seed_records for seed_records, _, _, _ in results))

# ==========================================
# 7. RINGKASAN (mean ± std dari semua seed)
# ==========================================
df_records = pd.DataFrame(records)
summary = df_records.groupby("variant", sort=False).agg(
    params=("params", "mean"), hemat_mean=("hemat", "mean"), hemat_std=("hemat", "std"), flops=("flops", "mean"),
    acc_mean=("acc", "mean"), acc_std=("acc", "std"), f1_mean=("f1", "mean"), f1_std=("f1", "std"),
    f1t_mean=("f1_tuned", "mean"), f1t_std=("f1_tuned", "std"), auc_mean=("auc", "mean"), auc_std=("auc", "std"),
    prauc_mean=("pr_auc", "mean"), prauc_std=("pr_auc", "std"), log_loss=("log_loss", "mean"), pos_rate=("pos_rate", "mean"),
    d_acc_mean=("d_acc", "mean"), d_acc_std=("d_acc", "std"), p_mcnemar=("p_mcnemar", "median"), p_ttest=("p_ttest", "median"),
    ranks=("ranks", "first"),
)
summary.to_csv("hasil_adult_final.csv")

n_seeds = len(SEEDS)
t_crit = stats.t.ppf(0.975, n_seeds - 1)

def tost_p(delta, margin):
    # Dua uji satu-sisi (TOST) pada selisih akurasi antar seed: H1 = |selisih| < margin
    if np.std(delta) == 0:
        return 0.0 if abs(np.mean(delta)) < margin else 1.0
    p_lower = stats.ttest_1samp(delta, -margin, alternative="greater").pvalue
    p_upper = stats.ttest_1samp(delta, margin, alternative="less").pvalue
    return max(p_lower, p_upper)

def paired_p(delta):
    return 1.0 if np.std(delta) == 0 else stats.ttest_1samp(delta, 0).pvalue

d_acc_wide = df_records.pivot(index="seed", columns="variant", values="d_acc").reindex(columns=summary.index)
acc_wide = df_records.pivot(index="seed", columns="variant", values="acc").reindex(columns=summary.index)
vs_proposed = acc_wide.sub(acc_wide[PROPOSED], axis=0)

def pm(mean_col, std_col, fmt="{:.2f}", sign=False):
    return summary[mean_col].map(("{:+.2f}" if sign else fmt).format) + " ± " + summary[std_col].map("{:.2f}".format)

ci_half = t_crit * summary["d_acc_std"] / np.sqrt(n_seeds)

table_budget = pd.DataFrame({
    "Params": summary["params"].round().astype(int),
    "Hemat (%)": pm("hemat_mean", "hemat_std", "{:.1f}"),
    "FLOPs eff (K)": (summary["flops"] / 1e3).map("{:.0f}".format),
    f"Rank (seed {SEEDS[0]})": summary["ranks"],
})

table_quality = pd.DataFrame({
    "Akurasi (%)": pm("acc_mean", "acc_std"),
    "F1@0.5 (%)": pm("f1_mean", "f1_std"),
    "F1@tuned (%)": pm("f1t_mean", "f1t_std"),
    "AUC (%)": pm("auc_mean", "auc_std"),
    "PR-AUC (%)": pm("prauc_mean", "prauc_std"),
    "LogLoss": summary["log_loss"].map("{:.4f}".format),
    "Pos-rate (%)": summary["pos_rate"].map("{:.2f}".format),
})

table_stats = pd.DataFrame({
    "dAkurasi vs Baseline (pp)": pm("d_acc_mean", "d_acc_std", sign=True),
    "CI95": "[" + (summary["d_acc_mean"] - ci_half).map("{:+.2f}".format) + ", " + (summary["d_acc_mean"] + ci_half).map("{:+.2f}".format) + "]",
    f"TOST p (+-{EQUIV_MARGIN}pp)": d_acc_wide.apply(lambda col: tost_p(col.values, EQUIV_MARGIN)).map("{:.4f}".format),
    "dAkurasi vs PROPOSED (pp)": vs_proposed.mean().map("{:+.2f}".format) + " ± " + vs_proposed.std().map("{:.2f}".format),
    "p vs PROPOSED": vs_proposed.apply(lambda col: paired_p(col.values)).map("{:.4f}".format),
    "McNemar p": summary["p_mcnemar"].map("{:.4f}".format),
    "t-test p": summary["p_ttest"].map("{:.4f}".format),
})

pd.set_option("display.width", 320)
pd.set_option("display.max_columns", 50)
for title, table in [("ANGGARAN PARAMETER & FLOPs", table_budget), ("KUALITAS", table_quality), ("EKUIVALENSI & PERBANDINGAN DENGAN PROPOSED", table_stats)]:
    print("\n" + "=" * 170)
    print(f"=== {title} (mean ± std dari {n_seeds} seed/split; p-value = median antar seed kecuali TOST/vs PROPOSED) ===")
    print("=" * 170)
    print(table.to_string())
print("Catatan: semua metode zero-shot (tanpa fine-tuning). CI95/TOST dari 5 seed (split saling tumpang tindih -> sedikit optimis). "
      "TOST p kecil = ekuivalen dalam margin; p>0.05 pada McNemar/t-test BUKAN bukti ekuivalen. Pruning unstructured: FLOPs efektif teoretis, latensi dense.")

print("\n" + "=" * 100)
print("=== RANK PROPOSED PER SEED ===")
print("=" * 100)
print(df_records[df_records["variant"] == PROPOSED][["seed", "ranks", "params", "hemat"]].round(2).to_string(index=False))

# ==========================================
# 8. KURVA TRADE-OFF (PARETO) AKURASI vs PARAMETER
# ==========================================
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.errorbar(summary["params"], summary["acc_mean"], yerr=summary["acc_std"], fmt="o", capsize=3)
for label, row in summary.iterrows():
    ax.annotate(label[:2], (row["params"], row["acc_mean"]), textcoords="offset points", xytext=(5, 5), fontsize=8)
ax.set_xscale("log")
ax.set_xlabel("Parameter efektif (log)")
ax.set_ylabel("Akurasi test (%)")
ax.set_title("Trade-off akurasi vs parameter (kode = nomor metode)")
fig.tight_layout()
fig.savefig("pareto_adult.png", dpi=200)

# ==========================================
# 9. BENCHMARK LATENSI (SEED PERTAMA, SINGLE-THREAD, DIANTERLEAVE ANTAR METODE)
# ==========================================
base_model_0, models_0, X_test_0 = results[0][1], results[0][2], results[0][3]
all_models = {"00. Baseline (Full)": base_model_0, **models_0}
batch_sizes = [1, 16, 64, 256, 1024, 4096, len(X_test_0)]

def bench_model(net, x_data, runs=100):
    net.eval()
    with torch.no_grad():
        for _ in range(15): _ = net(x_data) # Warmup
        timings = []
        for _ in range(runs):
            t0 = time.perf_counter()
            _ = net(x_data)
            t1 = time.perf_counter()
            timings.append((t1 - t0) * 1000)
    return np.mean(timings), np.std(timings), np.median(timings)

print("\nMengukur latensi semua metode (seed pertama)...")
latency_records = []
for batch_size, (label, net) in product(batch_sizes, all_models.items()):
    mean_ms, std_ms, median_ms = bench_model(net, X_test_0[:batch_size])
    latency_records.append({"batch_size": batch_size, "code": label[:2], "mean_ms": mean_ms, "std_ms": std_ms, "median_ms": median_ms})

df_latency = pd.DataFrame(latency_records)
baseline_median = df_latency[df_latency["code"] == "00"].set_index("batch_size")["median_ms"]
df_latency["speedup"] = df_latency["batch_size"].map(baseline_median) / df_latency["median_ms"]
df_latency["latency"] = df_latency["mean_ms"].map("{:.4f}".format) + " ± " + df_latency["std_ms"].map("{:.4f}".format)
df_latency.to_csv("hasil_adult_latensi.csv", index=False)

print("\nLegenda kode: " + " | ".join(f"{label[:2]} = {label[4:]}" for label in all_models))
print("\n" + "=" * 150)
print("=== SPEEDUP TERHADAP BASELINE (x, berbasis median; selisih < ~8% = noise) ===")
print("=" * 150)
print(df_latency.pivot(index="batch_size", columns="code", values="speedup").round(2).to_string())
print("\n=== LATENSI (ms, mean ± std) ===")
print(df_latency.pivot(index="batch_size", columns="code", values="latency").to_string())

Mengunduh dataset Adult Income dari OpenML...
Dimensi Input: 97 | Total data: 45,222 | Proporsi >50K: 0.248

[1/3] Melatih baseline dan mengompresi untuk 5 seed (15 epoch per seed, semua metode zero-shot)...
Seed 42 selesai
Seed 0 selesai
Seed 1 selesai
Seed 2 selesai
Seed 3 selesai

=== ANGGARAN PARAMETER & FLOPs (mean ± std dari 5 seed/split; p-value = median antar seed kecuali TOST/vs PROPOSED) ===
                                                       Params    Hemat (%) FLOPs eff (K)           Rank (seed 42)
variant                                                                                                          
00. Baseline (Full)                                    756737   0.0 ± 0.00          1510                        -
01. PROPOSED: SVD + ambang MP (kuantil q=0.25)         211707  72.0 ± 1.10           420    45/97, 89/512, 28/256
02. Ablation: ambang awal (rata-rata ekor)             407686  46.1 ± 0.94           812   60/97, 177/512, 80/256
03. Gavish-Donoho (anggar

# ABLATION 2

In [3]:
import time
import copy
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from functools import lru_cache
from itertools import chain, product
from scipy import stats
from scipy.integrate import cumulative_trapezoid
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (f1_score, accuracy_score, roc_auc_score, average_precision_score,
                             log_loss, precision_recall_curve)
warnings.filterwarnings('ignore')

# ==========================================
# 0. KONFIGURASI EKSPERIMEN
# ==========================================
QUICK = False                         # True = uji cepat (1 seed, epoch minimal); False = eksperimen penuh
SEEDS = [42] if QUICK else [42, 0, 1, 2, 3]
EQUIV_MARGIN = 0.5                    # margin ekuivalensi akurasi (pp). TETAPKAN SEBELUM MELIHAT HASIL
EVR_GRID = [0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.98, 0.99]   # kandidat EVR untuk seleksi via validasi (tanpa oracle)
SWEEP = {"08": 0.75, "09": 1.5, "10": 2.0}                # pengali ambang MP untuk kurva trade-off
LAYER_NAMES = ["fc1", "fc2", "fc3"]
BASE_WIDTHS = (1024, 512, 256)
MAX_ROWS = 100_000
BATCH_SIZE = 256

# Dua resep pelatihan baseline: standar (seperti sebelumnya) dan teregularisasi (dropout + weight decay + early stopping)
RECIPES = {
    "standar": dict(epochs=3 if QUICK else 15, lr=0.002, weight_decay=0.0, dropout=0.0, patience=None),
    "teregularisasi": dict(epochs=6 if QUICK else 40, lr=0.002, weight_decay=1e-2, dropout=0.2, patience=5),
}
FINETUNE = dict(epochs=1 if QUICK else 3, lr=3e-4, weight_decay=0.0, patience=None)

LATENCY_BATCHES = [1, 16, 64, 256, 1024]
LATENCY_RUNS, LATENCY_WARMUP = 60, 10
LATENCY_COLS = [f"lat_{bs}" for bs in LATENCY_BATCHES]

# Dataset biner tabular dari OpenML. ID/nama BELUM diverifikasi di lingkungan saya; yang gagal dimuat dilewati otomatis.
DATASET_SPECS = {
    "Adult Income": dict(name="adult", version=2),
    "Bank Marketing": dict(data_id=1461),
    "Credit Default (Taiwan)": dict(data_id=42477),
    "HIGGS (98k)": dict(data_id=23512),
}

LABELS = {
    "00": "Baseline (Full)",
    "01": "PROPOSED: SVD + ambang MP (zero-shot)",
    "02": "Gavish-Donoho (zero-shot)",
    "03": "EVR 85% (zero-shot)",
    "04": "EVR dicocokkan ke anggaran Proposed (oracle)",
    "05": "EVR dipilih via validasi (tanpa oracle)",
    "06": "Rank seragam dicocokkan ke anggaran Proposed",
    "07": "Magnitude pruning dicocokkan ke anggaran Proposed",
    "08": "Proposed, ambang x0.75",
    "09": "Proposed, ambang x1.5",
    "10": "Proposed, ambang x2.0",
    "11": "PROPOSED + fine-tune",
    "12": "EVR oracle + fine-tune",
    "13": "EVR validasi + fine-tune",
    "14": "Rank seragam + fine-tune",
    "15": "Pruning + fine-tune",
    "16": "Proposed x1.5 + fine-tune",
    "17": "Proposed x2.0 + fine-tune",
    "18": "Model kecil dari nol (anggaran = Proposed)",
}
NAME = {code: f"{code}. {label}" for code, label in LABELS.items()}
BASELINE_NAME, PROPOSED_NAME = NAME["00"], NAME["01"]
FT_SOURCES = {"11": "01", "12": "04", "13": "05", "14": "06", "15": "07", "16": "09", "17": "10"}

TRAIN_THREADS = torch.get_num_threads()    # training memakai semua core; benchmark latensi dikunci 1 thread

# ==========================================
# 1. DATASET
# ==========================================
def load_openml_binary(spec):
    X_raw, y_raw = fetch_openml(**spec, return_X_y=True, as_frame=True)
    data = pd.concat([X_raw, y_raw.rename("__target__")], axis=1).dropna()
    if len(data) > MAX_ROWS:
        data = data.sample(MAX_ROWS, random_state=0)
    target = data.pop("__target__").astype(str)
    positive = target.value_counts().idxmin()                 # kelas minoritas = kelas positif
    X = pd.get_dummies(data, drop_first=True).astype(float).values
    return X, (target == positive).astype(int).values

def load_all_datasets():
    loaded = {}
    for name, spec in DATASET_SPECS.items():
        try:
            X, y = load_openml_binary(spec)
            loaded[name] = (X, y)
            print(f"[OK]   {name}: {len(X):,} baris, {X.shape[1]} fitur, proporsi positif {y.mean():.3f}")
        except Exception as error:
            print(f"[SKIP] {name}: gagal dimuat ({type(error).__name__}: {str(error)[:80]})")
    return loaded

def prepare_data(X, y, seed):
    """Split train/val/test per seed; val (10% dari train) untuk menala ambang F1, early stopping, dan seleksi EVR;
    scaler hanya di-fit pada data train"""
    X_trainval, X_test, y_trainval, y_test = train_test_split(X, y, test_size=0.2, random_state=seed, stratify=y)
    X_train, X_val, y_train, y_val = train_test_split(X_trainval, y_trainval, test_size=0.1, random_state=seed, stratify=y_trainval)
    scaler = StandardScaler().fit(X_train)

    def to_x(array):
        return torch.tensor(scaler.transform(array), dtype=torch.float32)

    return {"X_train": to_x(X_train), "y_train": torch.tensor(y_train, dtype=torch.float32).unsqueeze(1),
            "X_val": to_x(X_val), "y_val": y_val, "y_val_tensor": torch.tensor(y_val, dtype=torch.float32).unsqueeze(1),
            "X_test": to_x(X_test), "y_test": y_test}

# ==========================================
# 2. ARSITEKTUR DAN PELATIHAN
# ==========================================
class WideMLP(nn.Module):
    def __init__(self, in_dim, dropout=0.0, widths=BASE_WIDTHS):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, widths[0])
        self.fc2 = nn.Linear(widths[0], widths[1])
        self.fc3 = nn.Linear(widths[1], widths[2])
        self.head = nn.Linear(widths[2], 1)
        self.act = nn.ReLU()
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        x = self.drop(self.act(self.fc1(x)))
        x = self.drop(self.act(self.fc2(x)))
        x = self.drop(self.act(self.fc3(x)))
        return self.head(x)

def train_model(model, data, recipe, seed):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW(model.parameters(), lr=recipe["lr"], weight_decay=recipe["weight_decay"])
    n_samples = len(data["X_train"])
    steps_per_epoch = n_samples // BATCH_SIZE

    # Seluruh urutan batch (semua epoch) disiapkan di muka, sehingga training cukup satu loop
    generator = torch.Generator().manual_seed(seed)
    batch_indices = torch.stack([torch.randperm(n_samples, generator=generator) for _ in range(recipe["epochs"])])
    batch_indices = batch_indices[:, :steps_per_epoch * BATCH_SIZE].reshape(-1, BATCH_SIZE)

    best_val, best_state, bad_epochs = float("inf"), None, 0
    model.train()
    for step, indices in enumerate(batch_indices):
        optimizer.zero_grad()
        criterion(model(data["X_train"][indices]), data["y_train"][indices]).backward()
        optimizer.step()

        if recipe["patience"] is not None and (step + 1) % steps_per_epoch == 0:     # early stopping via validation loss
            model.eval()
            with torch.no_grad():
                val_loss = criterion(model(data["X_val"]), data["y_val_tensor"]).item()
            model.train()
            if val_loss < best_val - 1e-4:
                best_val, best_state, bad_epochs = val_loss, copy.deepcopy(model.state_dict()), 0
            else:
                bad_epochs += 1
            if bad_epochs >= recipe["patience"]:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()

def dense_params(in_dim, widths):
    sizes = [in_dim, *widths]
    return sum(a * b + b for a, b in zip(sizes[:-1], sizes[1:])) + widths[-1] + 1

def scaled_widths(scale):
    return tuple(max(2, int(round(scale * w))) for w in BASE_WIDTHS)

def match_width(in_dim, target_params, iterations=40):
    # Lebar MLP dari nol: skala terbesar yang parameternya TIDAK melebihi anggaran Proposed
    low, high = 0.0, 1.0
    for _ in range(iterations):
        mid = (low + high) / 2
        if dense_params(in_dim, scaled_widths(mid)) > target_params:
            high = mid
        else:
            low = mid
    return low

# ==========================================
# 3. AMBANG / PEMILIHAN RANK (spektrum singular value = array numpy, menurun)
# ==========================================
@lru_cache(maxsize=None)
def mp_quantile(beta, q):
    """Kuantil ke-q distribusi Marchenko-Pastur (sigma=1, rasio aspek beta <= 1)"""
    lower, upper = (1 - np.sqrt(beta)) ** 2, (1 + np.sqrt(beta)) ** 2
    x = np.linspace(lower, upper, 20001)[1:-1]
    pdf = np.sqrt((upper - x) * (x - lower)) / (2 * np.pi * beta * x)
    cdf = cumulative_trapezoid(pdf, x, initial=0)
    return float(np.interp(q, cdf / cdf[-1], x))

def get_mp_threshold_tail(S, m, n, q=0.25):
    """[PROPOSED] Sigma dari pencocokan KUANTIL ke-q spektrum dengan kuantil MP, lalu tepi bulk sigma * (sqrt(m) + sqrt(n))"""
    small, large = sorted((m, n))
    beta = min(small / large, 0.999)
    sigma = np.quantile(S, q) / np.sqrt(large * mp_quantile(beta, q))
    return sigma * (np.sqrt(small) + np.sqrt(large))

def gavish_donoho_threshold(S, m, n):
    """[PEMBANDING] Optimal hard threshold Gavish-Donoho (2014) untuk noise tak diketahui"""
    small, large = sorted((m, n))
    beta = small / large
    return (0.56 * beta**3 - 0.95 * beta**2 + 1.82 * beta + 1.43) * np.median(S)

def get_spectra(model):
    layers = [getattr(model, name) for name in LAYER_NAMES]
    shapes = [tuple(layer.weight.shape) for layer in layers]
    spectra = [torch.linalg.svdvals(layer.weight.data).numpy() for layer in layers]
    return spectra, shapes

def ranks_by_threshold(spectra, shapes, threshold_fn, multiplier=1.0):
    return [max(1, int((S > multiplier * threshold_fn(S, m, n)).sum())) for S, (m, n) in zip(spectra, shapes)]

def ranks_evr(spectra, evr):
    return [min(len(S), int((np.cumsum(S**2) / (S**2).sum() < evr).sum()) + 1) for S in spectra]

def ranks_uniform(shapes, ratio):
    return [max(1, int(round(ratio * min(m, n)))) for m, n in shapes]

# ==========================================
# 4. KOMPRESI: SVD (RANK DIBERIKAN), MAGNITUDE PRUNING, DAN FINE-TUNING
# ==========================================
class FactoredLinear(nn.Module):
    def __init__(self, W_A, W_B, bias):
        super().__init__()
        k, in_f = W_A.shape
        out_f, _ = W_B.shape
        self.fc1 = nn.Linear(in_f, k, bias=False)
        self.fc2 = nn.Linear(k, out_f, bias=True)

        with torch.no_grad():
            self.fc1.weight.copy_(W_A)
            self.fc2.weight.copy_(W_B)
            self.fc2.bias.copy_(bias)

    def forward(self, x):
        return self.fc2(self.fc1(x))

def layer_params(m, n, k):
    # faktorisasi hanya dipakai jika menghemat parameter; selain itu layer asli dipertahankan
    return m * n + m if k * (m + n) >= m * n else k * (n + m) + m

def total_params(shapes, ranks, fixed_params):
    return sum(layer_params(m, n, k) for (m, n), k in zip(shapes, ranks)) + fixed_params

def effective_ranks_text(shapes, ranks):
    return ", ".join(f"{min(m, n) if k * (m + n) >= m * n else k}/{min(m, n)}" for (m, n), k in zip(shapes, ranks))

def compress_with_ranks(model, ranks):
    comp_net = nn.Sequential()
    for idx, (name, k) in enumerate(zip(LAYER_NAMES, ranks)):
        layer = getattr(model, name)
        W, b = layer.weight.data, layer.bias.data
        m, n = W.shape

        if k * (m + n) >= m * n:
            module = copy.deepcopy(layer)
        else:
            U, S, Vh = torch.linalg.svd(W, full_matrices=False)
            sqrt_S_k = torch.diag(torch.sqrt(S[:k]))
            module = FactoredLinear(sqrt_S_k @ Vh[:k, :], U[:, :k] @ sqrt_S_k, b)

        comp_net.add_module(f"layer{idx+1}", module)
        comp_net.add_module(f"relu{idx+1}", nn.ReLU())

    comp_net.add_module("head", copy.deepcopy(model.head))
    comp_net.eval()
    return comp_net

def match_budget(rank_fn, target_params, shapes, fixed_params, iterations=30):
    # Parameter total monoton naik terhadap pengatur rank (0..1): bisection mencari nilai terbesar
    # yang TIDAK melebihi anggaran parameter Proposed
    low, high = 0.0, 1.0
    for _ in range(iterations):
        mid = (low + high) / 2
        if total_params(shapes, rank_fn(mid), fixed_params) > target_params:
            high = mid
        else:
            low = mid
    return low

def magnitude_prune(model, ratio):
    model_pruned = copy.deepcopy(model)
    with torch.no_grad():
        for name in LAYER_NAMES:
            weight = getattr(model_pruned, name).weight
            weight.mul_((weight.abs() > torch.quantile(weight.abs().flatten(), ratio)).float())
    model_pruned.eval()
    return model_pruned

def prune_ratio_for_budget(model, target_params):
    weights_total = sum(getattr(model, name).weight.numel() for name in LAYER_NAMES)
    other_params = sum(p.numel() for p in model.parameters()) - weights_total
    return float(np.clip(1 - (target_params - other_params) / weights_total, 0.0, 0.999))

def attach_masks(model):
    # Bobot terpangkas tetap nol saat fine-tuning (gradiennya dinolkan)
    for name in LAYER_NAMES:
        weight = getattr(model, name).weight
        weight.register_hook(lambda grad, mask=(weight.data != 0).float(): grad * mask)

def finetune(model, data, seed, masked):
    model_ft = copy.deepcopy(model)
    if masked:
        attach_masks(model_ft)
    train_model(model_ft, data, FINETUNE, seed)
    return model_ft

# ==========================================
# 5. EVALUASI: KUALITAS, EFISIENSI, UJI STATISTIK
# ==========================================
def get_metrics(net, x_tensor):
    net.eval()
    with torch.no_grad():
        probs = torch.sigmoid(net(x_tensor).squeeze(1)).cpu().numpy()
    return (probs >= 0.5).astype(int), probs

def count_effective_params(net):
    return sum(int((p != 0).sum().item()) for p in net.parameters())

def count_effective_flops(net):
    # FLOPs per sampel = 2 x bobot non-nol pada semua layer Linear
    return 2 * sum(int((m.weight != 0).sum().item()) for m in net.modules() if isinstance(m, nn.Linear))

def tuned_threshold(y_val, probs_val):
    precision, recall, thresholds = precision_recall_curve(y_val, probs_val)
    f1_curve = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
    return thresholds[np.argmax(f1_curve)]

def val_accuracy(net, data):
    return float((get_metrics(net, data["X_val"])[0] == data["y_val"]).mean())

def select_evr_by_validation(base_model, spectra, shapes, fixed_params, data):
    # Tanpa oracle: EVR terkecil (parameter paling sedikit) yang akurasi validasinya turun <= margin dari baseline
    base_val_acc = val_accuracy(base_model, data)
    val_accs = {evr: val_accuracy(compress_with_ranks(base_model, ranks_evr(spectra, evr)), data) for evr in EVR_GRID}
    eligible = [evr for evr in EVR_GRID if val_accs[evr] >= base_val_acc - EQUIV_MARGIN / 100]
    if not eligible:
        return max(EVR_GRID)
    return min(eligible, key=lambda evr: total_params(shapes, ranks_evr(spectra, evr), fixed_params))

def timed_forward(net, x):
    start = time.perf_counter()
    net(x)
    return (time.perf_counter() - start) * 1000

def bench_median(net, x):
    net.eval()
    with torch.inference_mode():
        _ = [timed_forward(net, x) for _ in range(LATENCY_WARMUP)]
        timings = [timed_forward(net, x) for _ in range(LATENCY_RUNS)]
    return float(np.median(timings))

def measure_latencies(models, X_test):
    # Diukur dari seed pertama, single-thread, diselang-seling antar metode untuk tiap batch size
    previous_threads = torch.get_num_threads()
    torch.set_num_threads(1)
    medians = {(bs, code): bench_median(net, X_test[:bs]) for bs, (code, net) in product(LATENCY_BATCHES, models.items())}
    torch.set_num_threads(previous_threads)
    return medians

def evaluate_variant(variant_name, net, ranks_text, seed, data, reference, latency_by_bs):
    ref_preds, ref_probs, ref_params = reference
    y_test = data["y_test"]
    preds, probs = get_metrics(net, data["X_test"])
    probs_val = get_metrics(net, data["X_val"])[1]
    threshold = tuned_threshold(data["y_val"], probs_val)       # ambang dituning di validation set, per model

    acc, acc_ref = accuracy_score(y_test, preds), accuracy_score(y_test, ref_preds)
    f1, f1_ref = f1_score(y_test, preds), f1_score(y_test, ref_preds)

    b_mc = int(((ref_preds == y_test) & (preds != y_test)).sum())
    c_mc = int(((ref_preds != y_test) & (preds == y_test)).sum())
    p_mc = 1.0 if (b_mc + c_mc) == 0 else stats.binomtest(min(b_mc, c_mc), b_mc + c_mc, 0.5).pvalue

    params = count_effective_params(net)
    return {
        "variant": variant_name, "seed": seed, "params": params, "hemat": (1 - params / ref_params) * 100,
        "flops": count_effective_flops(net), "acc": acc * 100, "f1": f1 * 100,
        "f1_tuned": f1_score(y_test, (probs >= threshold).astype(int)) * 100,
        "auc": roc_auc_score(y_test, probs) * 100, "pr_auc": average_precision_score(y_test, probs) * 100,
        "log_loss": log_loss(y_test, np.clip(probs, 1e-7, 1 - 1e-7)), "pos_rate": preds.mean() * 100,
        "d_acc": (acc - acc_ref) * 100, "d_f1": (f1 - f1_ref) * 100, "p_mcnemar": p_mc, "ranks": ranks_text,
        **{f"lat_{bs}": latency_by_bs.get(bs, np.nan) for bs in LATENCY_BATCHES},
    }

# ==========================================
# 6. SATU PENGATURAN (DATASET x SEED x RESEP BASELINE)
# ==========================================
def run_setting(dataset_name, setting, data, in_dim, seed):
    recipe = RECIPES[setting]
    base_model = WideMLP(in_dim, dropout=recipe["dropout"])
    train_model(base_model, data, recipe, seed)
    preds_base, probs_base = get_metrics(base_model, data["X_test"])
    reference = (preds_base, probs_base, count_effective_params(base_model))

    spectra, shapes = get_spectra(base_model)
    fixed_params = sum(p.numel() for p in base_model.head.parameters())

    # Anggaran parameter Proposed menjadi acuan untuk semua pembanding "anggaran sama"
    ranks_proposed = ranks_by_threshold(spectra, shapes, get_mp_threshold_tail)
    target_params = total_params(shapes, ranks_proposed, fixed_params)
    evr_oracle = match_budget(lambda v: ranks_evr(spectra, v), target_params, shapes, fixed_params)
    ratio_oracle = match_budget(lambda v: ranks_uniform(shapes, v), target_params, shapes, fixed_params)
    evr_validated = select_evr_by_validation(base_model, spectra, shapes, fixed_params, data)

    rank_plan = {
        "01": ranks_proposed,
        "02": ranks_by_threshold(spectra, shapes, gavish_donoho_threshold),
        "03": ranks_evr(spectra, 0.85),
        "04": ranks_evr(spectra, evr_oracle),
        "05": ranks_evr(spectra, evr_validated),
        "06": ranks_uniform(shapes, ratio_oracle),
        **{code: ranks_by_threshold(spectra, shapes, get_mp_threshold_tail, mult) for code, mult in SWEEP.items()},
    }
    zero_shot = {code: compress_with_ranks(base_model, ranks) for code, ranks in rank_plan.items()}
    zero_shot["07"] = magnitude_prune(base_model, prune_ratio_for_budget(base_model, target_params))
    finetuned = {code: finetune(zero_shot[source], data, seed, masked=(source == "07")) for code, source in FT_SOURCES.items()}

    small_model = WideMLP(in_dim, dropout=recipe["dropout"], widths=scaled_widths(match_width(in_dim, target_params)))
    train_model(small_model, data, recipe, seed)

    all_models = dict(sorted({"00": base_model, **zero_shot, **finetuned, "18": small_model}.items()))
    structural = {**{code: code for code in all_models}, **FT_SOURCES}        # fine-tune memakai struktur (latensi) sumbernya
    latency = measure_latencies({code: net for code, net in all_models.items() if code not in FT_SOURCES}, data["X_test"]) if seed == SEEDS[0] else {}

    ranks_text = {code: effective_ranks_text(shapes, ranks) for code, ranks in rank_plan.items()}
    ranks_text.update({code: ranks_text[source] for code, source in FT_SOURCES.items() if source in ranks_text})
    return [
        {**evaluate_variant(NAME[code], net, ranks_text.get(code, "-"), seed, data, reference,
                            {bs: latency.get((bs, structural[code]), np.nan) for bs in LATENCY_BATCHES}),
         "dataset": dataset_name, "setting": setting}
        for code, net in all_models.items()
    ]

def run_dataset_seed(dataset_name, X, y, seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    data = prepare_data(X, y, seed)
    records = [run_setting(dataset_name, setting, data, X.shape[1], seed) for setting in RECIPES]
    print(f"Selesai: {dataset_name} | seed {seed}")
    return list(chain.from_iterable(records))

# ==========================================
# 7. RINGKASAN: STATISTIK, HOLM, TABEL
# ==========================================
def holm_adjust(p_values):
    p_values = np.asarray(p_values, dtype=float)
    order = np.argsort(p_values)
    adjusted, running = np.empty(len(p_values)), 0.0
    for rank, idx in enumerate(order):
        running = max(running, (len(p_values) - rank) * p_values[idx])
        adjusted[idx] = min(1.0, running)
    return adjusted

def holm_series(p_series, exclude):
    adjusted = p_series.copy()
    mask = ~p_series.index.isin(exclude)
    adjusted[mask] = holm_adjust(p_series[mask].values)
    return adjusted

def tost_p(delta, margin):
    # Dua uji satu-sisi (TOST) pada selisih akurasi antar seed: H1 = |selisih| < margin
    if np.std(delta) == 0:
        return 0.0 if abs(np.mean(delta)) < margin else 1.0
    p_lower = stats.ttest_1samp(delta, -margin, alternative="greater").pvalue
    p_upper = stats.ttest_1samp(delta, margin, alternative="less").pvalue
    return max(p_lower, p_upper)

def paired_p(delta):
    return 1.0 if np.std(delta) == 0 else stats.ttest_1samp(delta, 0).pvalue

def ci_text(frame, fmt):
    t_crit = stats.t.ppf(0.975, max(len(frame) - 1, 1))
    mean, half = frame.mean(), t_crit * frame.std() / np.sqrt(len(frame))
    return mean.map(fmt.format) + " [" + (mean - half).map(fmt.format) + ", " + (mean + half).map(fmt.format) + "]"

def pm(summary, mean_col, std_col, fmt="{:.2f}"):
    return summary[mean_col].map(fmt.format) + " ± " + summary[std_col].map("{:.2f}".format)

def build_tables(group):
    summary = group.groupby("variant", sort=False).agg(
        params=("params", "mean"), hemat_mean=("hemat", "mean"), hemat_std=("hemat", "std"), flops=("flops", "mean"),
        acc_mean=("acc", "mean"), acc_std=("acc", "std"), f1t_mean=("f1_tuned", "mean"), f1t_std=("f1_tuned", "std"),
        auc_mean=("auc", "mean"), auc_std=("auc", "std"), ranks=("ranks", "first"),
        **{col: (col, "first") for col in LATENCY_COLS},
    )

    def wide(column):
        return group.pivot(index="seed", columns="variant", values=column).reindex(columns=summary.index)

    d_acc, acc, log_loss_wide = wide("d_acc"), wide("acc"), wide("log_loss")
    d_log_loss = log_loss_wide.sub(log_loss_wide[BASELINE_NAME], axis=0)
    vs_proposed = acc.sub(acc[PROPOSED_NAME], axis=0)

    tost = holm_series(d_acc.apply(lambda col: tost_p(col.values, EQUIV_MARGIN)), exclude=[BASELINE_NAME])
    paired = holm_series(vs_proposed.apply(lambda col: paired_p(col.values)), exclude=[PROPOSED_NAME])
    speedup = summary[LATENCY_COLS].iloc[0] / summary[LATENCY_COLS]

    table_efficiency = pd.DataFrame({
        "Params": summary["params"].round().astype(int),
        "Hemat (%)": pm(summary, "hemat_mean", "hemat_std", "{:.1f}"),
        "FLOPs eff (K)": (summary["flops"] / 1e3).map("{:.0f}".format),
        f"Rank (seed {SEEDS[0]})": summary["ranks"],
        **{f"Spd bs={bs}": speedup[f"lat_{bs}"].map("{:.2f}x".format) for bs in LATENCY_BATCHES},
    })
    table_quality = pd.DataFrame({
        "Akurasi (%)": pm(summary, "acc_mean", "acc_std"),
        "F1@tuned (%)": pm(summary, "f1t_mean", "f1t_std"),
        "AUC (%)": pm(summary, "auc_mean", "auc_std"),
        "dAcc vs Base (pp) [CI95]": ci_text(d_acc, "{:+.2f}"),
        f"TOST p Holm (+-{EQUIV_MARGIN}pp)": tost.map("{:.4f}".format),
        "dAcc vs PROPOSED (pp)": vs_proposed.mean().map("{:+.2f}".format) + " ± " + vs_proposed.std().map("{:.2f}".format),
        "p Holm vs PROPOSED": paired.map("{:.4f}".format),
        "dLogLoss [CI95]": ci_text(d_log_loss, "{:+.4f}"),
    })
    return table_efficiency, table_quality, summary

def plot_pareto(ax, summary, title):
    codes = summary.index.str[:2].astype(int)
    styles = [("o", (codes < 11), "zero-shot"), ("^", (codes >= 11) & (codes < 18), "+ fine-tune"), ("s", (codes == 18), "kecil dari nol")]
    [ax.errorbar(summary["params"][mask], summary["acc_mean"][mask], yerr=summary["acc_std"][mask], fmt=marker, capsize=2, label=label)
     for marker, mask, label in styles]
    [ax.annotate(variant[:2], (row["params"], row["acc_mean"]), textcoords="offset points", xytext=(4, 4), fontsize=7)
     for variant, row in summary.iterrows()]
    ax.set_xscale("log")
    ax.set_title(title, fontsize=9)
    ax.set_xlabel("Parameter efektif (log)")
    ax.set_ylabel("Akurasi test (%)")
    ax.legend(fontsize=7)

# ==========================================
# 8. LOOP UTAMA DAN CETAK HASIL
# ==========================================
if __name__ == "__main__":
    datasets = load_all_datasets()
    print(f"\nMenjalankan {len(datasets)} dataset x {len(SEEDS)} seed x {len(RECIPES)} resep baseline (QUICK={QUICK})...")
    results = [run_dataset_seed(name, X, y, seed) for (name, (X, y)), seed in product(datasets.items(), SEEDS)]
    df_records = pd.DataFrame(list(chain.from_iterable(results)))
    df_records.to_csv("hasil_tabular_multi_per_seed.csv", index=False)

    pd.set_option("display.width", 400)
    pd.set_option("display.max_columns", 60)
    summaries, key_rows = {}, []
    for (dataset_name, setting), group in df_records.groupby(["dataset", "setting"], sort=False):
        table_efficiency, table_quality, summary = build_tables(group)
        summaries[(dataset_name, setting)] = summary
        print("\n" + "=" * 200)
        print(f"=== {dataset_name.upper()} | baseline {setting.upper()} (mean ± std dari {len(SEEDS)} seed; latensi dari seed {SEEDS[0]}, single-thread) ===")
        print("=" * 200)
        print(table_efficiency.to_string())
        print()
        print(table_quality.to_string())
        key_rows.append(table_efficiency[["Hemat (%)", f"Spd bs={LATENCY_BATCHES[-1]}"]].join(table_quality[["Akurasi (%)", "dAcc vs Base (pp) [CI95]", f"TOST p Holm (+-{EQUIV_MARGIN}pp)"]])
                        .loc[[NAME[c] for c in ("01", "11", "04", "05", "13", "06", "07", "15", "18")]]
                        .assign(dataset=dataset_name, setting=setting))

    print("\n" + "=" * 200)
    print("=== RINGKASAN LINTAS DATASET: PROPOSED vs PEMBANDING UTAMA ===")
    print("=" * 200)
    print(pd.concat(key_rows).set_index(["dataset", "setting"], append=True).reorder_levels(["dataset", "setting", "variant"]).to_string())

    fig, axes = plt.subplots(len(RECIPES), len(datasets), figsize=(5.5 * len(datasets), 4.5 * len(RECIPES)), squeeze=False)
    for (row, setting), (col, dataset_name) in product(enumerate(RECIPES), enumerate(datasets)):
        plot_pareto(axes[row][col], summaries[(dataset_name, setting)], f"{dataset_name} | {setting}")
    fig.tight_layout()
    fig.savefig("pareto_tabular_multi.png", dpi=200)

    print("\nCatatan: semua metode 'zero-shot' tanpa fine-tuning kecuali baris '+ fine-tune'. CI95/TOST dari seed (split saling tumpang tindih -> sedikit optimis); "
          "Holm dihitung per tabel. Pruning unstructured: FLOPs efektif teoretis, latensi dense. Model kecil dari nol memakai resep latih yang sama dengan baselinenya.")

[OK]   Adult Income: 45,222 baris, 97 fitur, proporsi positif 0.248
[OK]   Bank Marketing: 45,211 baris, 42 fitur, proporsi positif 0.117
[OK]   Credit Default (Taiwan): 30,000 baris, 23 fitur, proporsi positif 0.221
[OK]   HIGGS (98k): 98,049 baris, 28 fitur, proporsi positif 0.471

Menjalankan 4 dataset x 5 seed x 2 resep baseline (QUICK=False)...
Selesai: Adult Income | seed 42
Selesai: Adult Income | seed 0
Selesai: Adult Income | seed 1
Selesai: Adult Income | seed 2
Selesai: Adult Income | seed 3
Selesai: Bank Marketing | seed 42
Selesai: Bank Marketing | seed 0
Selesai: Bank Marketing | seed 1
Selesai: Bank Marketing | seed 2
Selesai: Bank Marketing | seed 3
Selesai: Credit Default (Taiwan) | seed 42
Selesai: Credit Default (Taiwan) | seed 0
Selesai: Credit Default (Taiwan) | seed 1
Selesai: Credit Default (Taiwan) | seed 2
Selesai: Credit Default (Taiwan) | seed 3
Selesai: HIGGS (98k) | seed 42
Selesai: HIGGS (98k) | seed 0
Selesai: HIGGS (98k) | seed 1
Selesai: HIGGS (98k) | s